<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/09-etica-el-modelo-que-discrimina/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Solución 2.9 · Ética II: el modelo que aprendió a discriminar

**Serie:** Del Excel al Machine Learning · **Temporada 2** · 🎬 Video próximamente

Estas son las respuestas del ejercicio. Si la tuya es distinta pero da el mismo resultado, ¡también está bien! En programación suele haber más de una forma de llegar a la respuesta.

### Prepara los datos

Ejecuta estas celdas. Dejan listas `entreno`, `prueba`, `base`, `indicadores`, `sorteo` y la función. Debe darte `72` en el sorteo.

**Este ejercicio solo cuenta tasas de un archivo inventado.** No armes una lista de personas elegidas.

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def mujeres_que_llamaria(entreno, prueba, columnas):
    """Solo cuenta: a cuántas mujeres llamaría un modelo que imita al reclutador, en cada cargo."""
    x_entreno = pd.get_dummies(entreno[columnas], dtype=int)
    x_prueba = pd.get_dummies(prueba[columnas], dtype=int).reindex(
        columns=x_entreno.columns, fill_value=0
    )
    modelo = make_pipeline(StandardScaler(), LogisticRegression())
    modelo.fit(x_entreno, entreno["entrevistado"])
    probabilidad = pd.Series(modelo.predict_proba(x_prueba)[:, 1], index=prueba.index)
    mujeres = 0
    for _, grupo in prueba.groupby("cargo"):
        cupo = int(grupo["entrevistado"].sum())
        orden = probabilidad[grupo.index].sort_values(ascending=False, kind="stable")
        mujeres += (prueba.loc[orden.index[:cupo], "sexo"] == "F").sum()
    return int(mujeres)

In [3]:
postulaciones = pd.read_csv(DATOS + "postulaciones.csv")

entreno = postulaciones[postulaciones["fecha"] < "2025-01-01"]
prueba = postulaciones[postulaciones["fecha"] >= "2025-01-01"]

base = ["cargo", "anios_experiencia", "puntaje_prueba"]
indicadores = ["meses_sin_empleo", "puede_viajar"]

sorteo = 0
for _, grupo in prueba.groupby("cargo"):
    sorteo += grupo["entrevistado"].sum() * (grupo["sexo"] == "F").mean()

print(round(sorteo))

72


### 1. ¿Qué aprende?

Responde en un comentario. Imagina que un modelo acierta el 90 % de `entrevistado`. ¿Significa que elige bien a las personas? ¿Por qué?

In [4]:
# No. Significa que se parece al reclutador, incluido su sesgo. No sabemos cómo habría
# trabajado quien no fue llamado, y la etiqueta no mide desempeño.

### 2. Solo sin `meses_sin_empleo`

Calcula cuántas mujeres llamaría un modelo que ve `base` y `puede_viajar`, sin `sexo` ni `meses_sin_empleo`. Debe dar `53`.

In [5]:
mujeres_que_llamaria(entreno, prueba, base + ["puede_viajar"])

53

### 3. Solo sin `puede_viajar`

Ahora con `base` y `meses_sin_empleo`. Debe dar `50`. ¿Cuál de los dos indicadores pesa más en estos datos?

In [6]:
mujeres_que_llamaria(entreno, prueba, base + ["meses_sin_empleo"])

# Quedan muy cerca: los dos pesan. Es de estos datos, no una regla.

50

### 4. Solo por puntaje

Calcula cuántas mujeres se llamarían con un modelo que solo ve el cargo y el puntaje, usando `["cargo", "puntaje_prueba"]`. Debe dar `67`.

Responde en un comentario: ¿esto prueba que la prueba es justa en la vida real?

In [7]:
mujeres_que_llamaria(entreno, prueba, ["cargo", "puntaje_prueba"])

# No. Aquí el puntaje se programó igual para todos. Una prueba real podría estar sesgada,
# y hay que comprobarlo. Y este resultado cambia con el azar de pocos casos.

67

### 5. Por cargo

Calcula cuántas mujeres y cuántos hombres postularon a cada cargo en `prueba`. ¿Por qué la referencia es un sorteo **dentro de cada cargo**, y no el total?

In [8]:
pd.crosstab(prueba["cargo"], prueba["sexo"])

# Porque postulan distinto a cada cargo. El total mezclaría esa diferencia con la del reclutador.

sexo,F,M
cargo,,
Almacenero,49,164
Asistente administrativo,97,50
Repartidor,27,162
Vendedor,98,136


### 6. La reunión

Responde en un comentario. Tu gerente propone: *"quitamos `sexo` y ya es un modelo justo. Lo activamos el lunes"*. Escribe tres frases para responderle, sin tecnicismos.

In [9]:
# Quitar esa columna no lo arregla: otras columnas la delatan, y aquí lo medimos.
# El modelo aprende a quién se llamaba, no quién trabaja bien, así que copia el problema.
# Propongo criterios escritos, revisar las tasas por cargo, y un responsable, antes de automatizar.

## 🎉 ¡Bien hecho!

Nos vemos en el **episodio 2.10: Miniproyecto: el reporte de la mañana**.

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0